# Day 2 — Applied Lab: Multi-Agent CX with Persistent Memory
### Agentic Customer Experience Specialisation · Post-lunch session (4h)

**What we're building today:** a supervisor agent that routes customer questions to the right
specialist instead of one agent trying to know everything; a memory layer that survives a channel
switch, so a customer who starts on chat and follows up on SMS doesn't have to repeat themselves;
and an agent-assist surface where the agent drafts, a human decides, and every decision is logged
for QA. By the end, you'll have a routed multi-agent flow, a memory-enabled agent proven to persist
context across two channels, and an assist surface with a QA hook watching it.

**How the six post-lunch topics map onto this notebook:**

| # | Topic | Where |
|---|---|---|
| 1 | Supervisor orchestration | Lab 1, Steps 3–4 |
| 2 | Specialist agents | Lab 1, Steps 1–2 |
| 3 | Agent-assist mode | Lab 3, Step 7 |
| 4 | Memory architecture | Lab 2, Step 5 |
| 5 | Channel adapters | Lab 2, Step 6 |
| 6 | QA hooks | Lab 3, Step 8 |

**Demo lanes:** Insurance (Lab 1 — supervisor + 2 specialists), Banking (Lab 2 — memory across
channels, **this is today's ship line**), Retail (Lab 3 — agent-assist + QA hooks). Unlike Day 1,
where one lane carried all six topics, today's three labs each live in a different vertical on
purpose — routing, memory, and assist are each easiest to see fresh, in a domain that wasn't
already carrying Day 1's baggage.

**Continuity with Day 1:** Lab 1's policy specialist reuses yesterday's Insurance knowledge base
verbatim. Yesterday, one agent held the whole policy KB *and* had to reason about claim status
questions it had no real data for. Today, the same KB becomes one specialist's tool, and a new
claims database becomes another's — the split itself is the lesson.

**How this notebook is organized:** same conventions as Day 1 — each code cell is preceded by an
explanation of *why it's built this way*, not just what it does.

| Marker | Meaning |
|---|---|
| ▶ Run this | A cell you execute and observe the output of |
| ✅ Checkpoint | A natural save point — the system is in a working, shippable state |
| 🔍 Try it yourself | A prompt to test with your own input, not just the example given |

A few cells are built to demonstrate a specific failure mode on purpose (a supervisor that
free-answers instead of routing, an agent with amnesia after a channel switch, an ungrounded assist
suggestion). The explanation before each one says so directly, and says what to expect.


## Setup

Same prerequisites as Day 1 — if you're continuing in the same environment, this cell is a
confirmation, not a fresh install.

1. Node.js + npm
2. The Claude Code CLI: `npm install -g @anthropic-ai/claude-code`
3. Python 3.10+, then `pip install claude-agent-sdk anthropic`
4. `export ANTHROPIC_API_KEY=your-key`

▶ **Run this** — confirms the environment and pulls in two more names we didn't need yesterday:
`AssistantMessage` and `TextBlock`. Yesterday's `ask()` helper just printed whatever the SDK handed
back; today, Lab 1's supervisor needs to *capture* a specialist's answer as a plain string and hand
it to another agent, so we need to reach into the message objects and pull the text out ourselves.


In [ ]:
# Setup — run this first.
import os, json, time, shutil

# Same runtime dependency as Day 1: this SDK shells out to the Claude Code CLI, it's
# not a direct HTTP client. Two names are new today —
#   AssistantMessage, TextBlock -> content-block types used to pull plain text back out
#   of a message stream. Day 1's ask() only ever PRINTED messages for a human to read;
#   Lab 1's _run_specialist() below needs to CAPTURE a specialist's final answer as a
#   string and hand it back as a tool result, which means filtering the message stream
#   for these two types instead of just printing everything.
from claude_agent_sdk import (
    tool, create_sdk_mcp_server, ClaudeAgentOptions, ClaudeSDKClient,
    AssistantMessage, TextBlock,
)

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing"
assert shutil.which("claude"), (
    "Claude Code CLI not found on PATH — run: npm install -g @anthropic-ai/claude-code"
)

BUILTIN_LOCKDOWN = ["Bash", "Read", "Write", "Edit", "Glob", "Grep"]

async def ask(options, question):
    """One-shot helper for single-turn tests — same as Day 1."""
    # Opens a session, sends ONE message, prints everything, closes the session.
    # Every call to ask() is a brand-new ClaudeSDKClient — nothing survives between
    # calls. Lab 2's two-session amnesia demo depends on this: session 1 and session 2
    # are two SEPARATE ask() calls specifically because that's what a real channel
    # switch looks like (a new session, not a continued one).
    async with ClaudeSDKClient(options=options) as client:
        await client.query(question)
        async for message in client.receive_response():
            print(message)

print("Environment ready.")


## Architecture — what changes when there's more than one agent

Day 1's diagram had one `ClaudeSDKClient`, one model, one in-process tool server. Today's supervisor
pattern doesn't add a new SDK primitive — it reuses the exact same one, twice: once for the
supervisor, once (or twice) more for whichever specialist gets called, nested inside a tool.

```
 customer message
        │
        ▼
┌─────────────────────────┐
│  Supervisor ClaudeSDKClient │   Its allowed_tools are NOT domain tools —
│  system_prompt: routing     │   they're ask_policy_specialist / ask_claims_specialist.
│  rules only, no KB access   │   The supervisor cannot call kb_search or claim_lookup itself.
└────────────┬─────────────┘
             │ tool_use: ask_claims_specialist(question)
             ▼
┌─────────────────────────┐
│  ask_claims_specialist       │   A plain @tool function. Its body opens a
│  (a Python function,         │   BRAND NEW ClaudeSDKClient, with its own
│  not another agent object)   │   ClaudeAgentOptions (own system_prompt, own
└────────────┬─────────────┘   tools — e.g. claim_lookup), runs one query,
             │ runs a nested session               and returns the specialist's
             ▼                                       final text as a tool_result.
┌─────────────────────────┐
│  Claims specialist            │
│  ClaudeSDKClient session      │
│  (its own tool: claim_lookup) │
└─────────────────────────┘
```

**The load-bearing idea:** a "specialist agent" is not a new SDK concept — it's an ordinary agent
(its own `ClaudeAgentOptions`, its own tools, its own persona if you want one) that happens to be
*invoked from inside a tool function* instead of directly from your test code. From the supervisor's
point of view, `ask_claims_specialist` looks exactly like `kb_search` did yesterday: a name, a
description, an input schema, a text result. What's different is only what runs inside it.

**Why the supervisor gets zero domain tools:** if `kb_search` and `claim_lookup` were also wired
into the supervisor's `allowed_tools`, there'd be nothing forcing it to route at all — it could just
answer everything itself, same failure mode as Day 1's ungrounded agent, one level up. Restricting
the supervisor to *only* the two specialist-dispatch tools is what makes routing structural rather
than a suggestion in a system prompt.

**A real SDK feature we're naming and not using today:** `ClaudeAgentOptions` has an `agents` field
(`dict[str, AgentDefinition]`) that wires named subagents into the CLI's own built-in dispatch tool
— the officially supported version of the pattern above. It's out of scope today for the same reason
Day 1 named `permission_mode` and `hooks` without using them: the manual version below makes the
*mechanism* visible — a specialist is just an agent, a dispatch is just a tool call — before showing
the shortcut that hides it. Worth a mention live if asked "is this how you'd really do it in
production" — the honest answer is: the concept is real and it's built into the SDK, today's version
is the teaching-visible path to the same place.

Nothing to run yet — Step 1 starts building the specialists.


## Lab 1 — Insurance: a supervisor routing to two specialists

**Ship criterion:** a supervisor agent that correctly routes a policy-coverage question to the
policy specialist, a claim-status question to the claims specialist, and — for a question that
needs both — calls both and synthesizes one combined answer, demonstrated by the agent itself
driving the routing decision, not by you calling the right function directly.

### Step 1 — The policy specialist

This is yesterday's knowledge base and `kb_search` tool, unchanged. The only thing that's new is
what it's *for*: yesterday it was the whole agent's only tool. Today it's one specialist's only
tool, and the specialist has no idea a supervisor or a claims specialist even exists — it just
answers policy questions when asked, exactly like Day 1's grounded agent did.

▶ **Run this** and confirm `policy_specialist_options` is defined. Nothing calls it yet.


In [ ]:
policy_chunks = [
    {"id": "POL-4.2", "text": "Comprehensive coverage does not include rental vehicle "
     "reimbursement unless Rider R-12 (Rental Reimbursement) has been purchased separately."},
    {"id": "POL-4.3", "text": "Collision coverage applies to damages resulting from an accident "
     "involving the insured vehicle and does not extend to third-party rental vehicles."},
    {"id": "POL-9.1", "text": "Rider R-12 provides up to INR 1,500/day for rental vehicle costs, "
     "capped at 30 days, while the insured vehicle is under repair due to a covered claim."},
    {"id": "POL-2.5", "text": "A covered claim requires an incident report filed within 7 days "
     "of the event and, for collision claims, a repair estimate from an approved garage."},
]
# Identical to Day 1's KB — this is the continuity point: yesterday this WAS the whole
# agent's knowledge; today it becomes one specialist's tool, nothing else about it changes.

def score(query: str, text: str) -> float:
    # Jaccard-style word overlap, same stand-in as Day 1 — swap for a real embedding
    # model + vector store in production; everything downstream keeps the same shape.
    q = set(query.lower().split())
    t = set(text.lower().split())
    return len(q & t) / max(len(q), 1)

def search(query: str, top_k: int = 3):
    ranked = sorted(policy_chunks, key=lambda c: score(query, c["text"]), reverse=True)
    return ranked[:top_k]

@tool(
    "kb_search",
    "Search the insurance policy knowledge base for clauses relevant to a customer's question. "
    "Always call this before answering any question about what is or isn't covered.",
    {"query": str},
)
async def kb_search(args):
    results = search(args["query"])
    formatted = "\n".join(f"[{r['id']}] {r['text']}" for r in results)
    return {"content": [{"type": "text", "text": formatted}]}   # the MCP tool-result contract

# One ClaudeAgentOptions = one complete, independent agent — this IS "the policy specialist,"
# there is no separate "specialist" class or object. Note it has NO IDEA a supervisor or a
# claims specialist exists; it just answers policy questions when asked, same as Day 1's
# grounded agent did.
policy_specialist_options = ClaudeAgentOptions(
    system_prompt=(
        "You are the POLICY specialist. For ANY question about coverage, exclusions, or policy "
        "terms, you MUST call kb_search first and answer only from the returned clauses. Always "
        "cite the clause id(s) you used, in the form [POL-x.x]. If the retrieved clauses don't "
        "fully answer the question, say so plainly — never fill gaps from general knowledge."
    ),
    mcp_servers={"policy_tools": create_sdk_mcp_server(
        name="policy_tools", version="1.0.0", tools=[kb_search],
    )},
    allowed_tools=["mcp__policy_tools__kb_search"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

print("policy_specialist_options ready.")


### Step 2 — The claims specialist

A second, independent specialist with its own tool: `claim_lookup`, backed by a small mock claims
database. Note `CLM-1077` — it's denied, and the denial reason references POL-2.5, the 7-day
incident-report window from yesterday's KB. That's not incidental: it's the seed of Step 4's
multi-intent test, where a single customer question genuinely needs both specialists to answer
completely.

**Why this is a separate specialist and not just another tool on the policy agent:** the policy
specialist's system prompt is built entirely around "ground every answer in kb_search." Bolting a
second, unrelated tool onto it means every answer now has to reason about *which* tool applies —
exactly the ambiguity a supervisor exists to resolve one level up, instead of pushing it down into
a single, more confused agent.

▶ **Run this** and confirm `claims_specialist_options` is defined.


In [ ]:
claims_db = {
    "CLM-1000": {"policy_id": "POL-100", "status": "approved", "filed": "2026-07-14",
                 "amount": 25000},
    "CLM-1042": {"policy_id": "POL-233", "status": "under review", "filed": "2026-07-16",
                 "amount": 8000},
    # CLM-1077's denial_reason references POL-2.5 — a policy rule, not a claims fact.
    # This is deliberate: it's the seed of Step 4's multi-intent test, where one question
    # genuinely needs both specialists to answer completely.
    "CLM-1077": {"policy_id": "POL-509", "status": "denied", "filed": "2026-07-09",
                 "amount": 15000,
                 "denial_reason": "Incident report filed after the 7-day window (POL-2.5)."},
}

@tool(
    "claim_lookup",
    "Look up the status of an existing insurance claim by its claim id. Always call this before "
    "answering any question about a specific claim's status, amount, or outcome.",
    {"claim_id": str},
)
async def claim_lookup(args):
    record = claims_db.get(args["claim_id"])
    if not record:
        return {"content": [{"type": "text", "text": f"No claim found with id {args['claim_id']}."}]}
    return {"content": [{"type": "text", "text": json.dumps(record)}]}

# A second, fully independent agent — its own tool, its own grounding rule, no awareness
# of policy_specialist_options at all. Scoped to ONE tool on purpose: bolting claim_lookup
# onto the policy specialist would force every answer to reason about which tool applies —
# exactly the ambiguity a supervisor exists to resolve one level up.
claims_specialist_options = ClaudeAgentOptions(
    system_prompt=(
        "You are the CLAIMS specialist. For ANY question about an existing claim's status, "
        "amount, or outcome, you MUST call claim_lookup first and answer only from the returned "
        "record. If a claim was denied, state the denial_reason plainly. Never guess a claim's "
        "status without looking it up."
    ),
    mcp_servers={"claims_tools": create_sdk_mcp_server(
        name="claims_tools", version="1.0.0", tools=[claim_lookup],
    )},
    allowed_tools=["mcp__claims_tools__claim_lookup"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

print("claims_specialist_options ready.")


### Step 3 — A supervisor that doesn't route

`_run_specialist` is the piece that makes a specialist callable as a tool: it opens a fresh
`ClaudeSDKClient` with the specialist's own options, runs one query, and walks the returned messages
pulling out only the `TextBlock` content of any `AssistantMessage` — that's the specialist's final
answer, as a plain string, ready to hand back as a `tool_result`.

`weak_supervisor_options` wires both specialists in as tools, but the system prompt is just *"You
are a customer service supervisor for an insurance company. You can consult a policy specialist or
a claims specialist if needed."* — "if needed" is exactly Day 1 Step 3's gap in a new shape: nothing
*requires* the model to route rather than answer from its own general knowledge.

**What to expect, and why:** ask *"My claim CLM-1077 got denied — why, and what's the actual filing
deadline?"* The model may answer part of this from general insight ("claims are often denied for
missing documentation") without ever calling `claim_lookup`, and never surface the real POL-2.5
deadline at all. This is the same engineered failure as Day 1 — a capable agent with the right tools
available still needs to be told, explicitly, that it must use them.

🔍 **Try it yourself** with the exact question above before moving to the fix — this failure needs
to reproduce for Step 4 to land.


In [ ]:
async def _run_specialist(options: ClaudeAgentOptions, question: str) -> str:
    """Runs one specialist agent end-to-end and returns its final text reply as a plain
    string. This is what lets a supervisor treat an entire nested agent as one tool call —
    everything the specialist does internally (its own tool calls, its own reasoning) stays
    inside this function; the supervisor only ever sees the string that comes out."""
    reply_parts = []
    # A brand-new ClaudeSDKClient, same as ANY agent call — this is the load-bearing point
    # of the whole lab: a "specialist" is not a new SDK concept, it's this exact object,
    # just invoked from inside a @tool function instead of from your own test code.
    async with ClaudeSDKClient(options=options) as client:
        await client.query(question)
        async for message in client.receive_response():
            # Filter the message stream down to only the final assistant TEXT — tool_use,
            # tool_result, and any intermediate messages the specialist produced along the
            # way are deliberately dropped here; the supervisor only needs the answer.
            # Resetting on every AssistantMessage (not appending across messages) is what
            # keeps this to the LAST one — a specialist that emits preamble text ("Let me
            # check that") before a tool call would otherwise leak into the returned string.
            if isinstance(message, AssistantMessage):
                reply_parts = []
                for block in message.content:
                    if isinstance(block, TextBlock):
                        reply_parts.append(block.text)
    return "".join(reply_parts)

@tool(
    "ask_policy_specialist",
    "Route a question about policy coverage, exclusions, or terms to the policy specialist agent.",
    {"question": str},
)
async def ask_policy_specialist(args):
    # From the supervisor's point of view this looks exactly like Day 1's kb_search did:
    # a name, a description, one string arg, one string result. What's different is only
    # what runs INSIDE it — a whole nested agent conversation, not a dict lookup.
    answer = await _run_specialist(policy_specialist_options, args["question"])
    return {"content": [{"type": "text", "text": answer}]}

@tool(
    "ask_claims_specialist",
    "Route a question about an existing claim's status, amount, or outcome to the claims "
    "specialist agent.",
    {"question": str},
)
async def ask_claims_specialist(args):
    answer = await _run_specialist(claims_specialist_options, args["question"])
    return {"content": [{"type": "text", "text": answer}]}

supervisor_tools_server = create_sdk_mcp_server(
    name="supervisor_tools", version="1.0.0",
    tools=[ask_policy_specialist, ask_claims_specialist],
)

# Note what's NOT in allowed_tools: no kb_search, no claim_lookup. The supervisor can
# ONLY dispatch to a specialist, never answer a domain question itself — same guardrail
# move as Day 1's BUILTIN_LOCKDOWN, a capability it doesn't have can't be misused no
# matter how the prompt is worded.
weak_supervisor_options = ClaudeAgentOptions(
    # "if needed" is the bug under test — nothing here REQUIRES routing over answering
    # from general knowledge, same shape of gap as Day 1's weak_options.
    system_prompt=(
        "You are a customer service supervisor for an insurance company. You can consult a "
        "policy specialist or a claims specialist if needed."
    ),
    mcp_servers={"supervisor_tools": supervisor_tools_server},
    allowed_tools=["mcp__supervisor_tools__ask_policy_specialist",
                   "mcp__supervisor_tools__ask_claims_specialist"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

await ask(weak_supervisor_options,
          "My claim CLM-1077 got denied — why, and what's the actual filing deadline?")


### Step 4 — Fixing the routing, and a question that needs both specialists

`supervisor_options` closes the gap with an explicit, structural routing rule: **the supervisor
never answers a policy or claims question from its own knowledge — it always delegates.** Read the
prompt clause by clause:

- *"You do not answer questions about coverage or claims yourself"* — removes the option to skip
  routing, same move as Day 1's "you MUST call kb_search first."
- *"call ask_policy_specialist"* / *"call ask_claims_specialist"* — the two routing rules, named
  explicitly rather than left to "if needed."
- *"If a question needs both, call both and synthesize one combined answer citing what each
  specialist said"* — this is the multi-intent case: the denied-claim question genuinely needs the
  claims specialist (what happened, the denial reason) **and** the policy specialist (what the
  actual rule is), and a good supervisor doesn't pick one arbitrarily.

Run the identical question from Step 3 against `supervisor_options`. Expect **both**
`ask_claims_specialist` and `ask_policy_specialist` to be called, and a combined answer that states
the claim was denied for a late incident report *and* confirms the 7-day window from POL-2.5.

**Why checking "both tools were called" matters more than checking the final text reads
correctly:** a supervisor that calls only one specialist and pads the rest with a plausible-sounding
guess can still produce an answer that *reads* complete. The only way to know it's actually
grounded in both specialists' real data is to check that both were called — the same lesson as Day
1's citation check: present-and-correct, not just present.

🔍 **Try it yourself:** ask a question that only needs one specialist (e.g. *"What's the status of
CLM-1000?"*) and confirm the supervisor calls only `ask_claims_specialist`, not both — routing
correctly includes knowing when *not* to call a specialist, not just when to.

✅ **Checkpoint — Lab 1 complete.** A working supervisor + 2 specialists: routes correctly,
synthesizes when a question spans both. Save the notebook here.


In [ ]:
supervisor_options = ClaudeAgentOptions(
    system_prompt=(
        "You are a customer service supervisor for an insurance company. You do not answer "
        # Removes the "if needed" escape hatch from the weak version — routing becomes
        # mandatory, not optional.
        "questions about coverage or claims yourself. For ANY question about policy coverage, "
        "exclusions, or terms, call ask_policy_specialist. For ANY question about an existing "
        "claim's status, amount, or outcome, call ask_claims_specialist. If a single question "
        # This is the multi-intent clause — the actual fix for the CLM-1077 test case, which
        # needs BOTH specialists to answer completely.
        "needs both (for example, why a claim was denied AND what the underlying policy rule "
        "is), call both specialists and synthesize one combined answer that cites what each one "
        "said. Never guess at coverage rules or claim details yourself."
    ),
    mcp_servers={"supervisor_tools": supervisor_tools_server},
    allowed_tools=["mcp__supervisor_tools__ask_policy_specialist",
                   "mcp__supervisor_tools__ask_claims_specialist"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Same question as the weak version — the real check is whether BOTH
# ask_policy_specialist and ask_claims_specialist actually get called this time, not
# just whether the final sentence sounds plausible.
await ask(supervisor_options,
          "My claim CLM-1077 got denied — why, and what's the actual filing deadline?")


---

## Lab 2 — Banking: memory that survives a channel switch

**Ship criterion — this is today's headline ship line:** a customer opens a dispute on one channel,
follows up on a different channel in a new session, and the agent picks the conversation back up
without asking the customer to repeat what they already said. Demonstrated by two separate
`ClaudeSDKClient` sessions, not by manually feeding the second session the first one's transcript.

**The fact this lab is built around:** a `ClaudeSDKClient` session only remembers what happened
*inside that session*. Switch channels — chat to SMS, web to voice — and in reality that's very
often a brand new session, sometimes even a different backend entirely. If memory only lives in the
model's context window, it evaporates at exactly the moment a real omnichannel customer needs it
most: mid-problem, switching channels because chat is slow or they're now on the move.

### Step 5 — The memory store

Two structures, kept deliberately separate:

- **Episodic memory** — a timestamped log of *events*: "customer disputed a $45 charge on card
  ending 9931, via chat." Raw, specific, grows over time.
- **Semantic memory** — durable *facts about the customer*, distilled rather than logged: "repeat
  disputer — 3 disputes in the last 90 days." Not a record of what happened, a conclusion drawn from
  what's happened.

**Why the split matters, not just as a naming exercise:** episodic memory answers "what did this
customer already tell us" — the thing that stops a channel switch from feeling like starting over.
Semantic memory answers "what do we know about this customer in general" — the thing a risk model or
a routing decision would actually use. Conflating them means either the episodic log gets scanned
every time a general fact is needed (slow, and re-derives the same conclusion repeatedly), or
semantic facts get buried as one-off log entries no one distills.

▶ **Run this** and confirm all three memory tools print as registered.


In [ ]:
memory_store = {}   # user_id -> {"episodic": [...], "semantic": {...}} — swap for real storage in production

def _bucket(user_id: str):
    # setdefault: first call for a given user_id creates the empty shape, every later
    # call just returns the same dict object — this is the ENTIRE persistence mechanism.
    # It lives outside any one ClaudeSDKClient session, which is the whole point.
    return memory_store.setdefault(user_id, {"episodic": [], "semantic": {}})

@tool(
    "remember_episode",
    "Record a specific event from this conversation to the customer's episodic memory — e.g. "
    "a dispute filed, a question asked, an action taken. Call this before ending any turn where "
    "the customer told you something worth recalling later.",
    {"user_id": str, "channel": str, "note": str},
)
async def remember_episode(args):
    bucket = _bucket(args["user_id"])
    bucket["episodic"].append({"channel": args["channel"], "note": args["note"], "ts": time.time()})
    return {"content": [{"type": "text", "text": "Episode recorded."}]}

@tool(
    "remember_fact",
    "Record or update a durable fact about the customer that should persist across every future "
    "conversation, regardless of channel — not a one-off event, a standing conclusion.",
    {"user_id": str, "key": str, "value": str},
)
async def remember_fact(args):
    # Note the shape difference from remember_episode: a KEYED upsert (bucket["semantic"][key]),
    # not an append. Calling this twice with the same key OVERWRITES, it doesn't grow a list —
    # that's what makes it a standing fact instead of another log entry.
    bucket = _bucket(args["user_id"])
    bucket["semantic"][args["key"]] = args["value"]
    return {"content": [{"type": "text", "text": f"Fact recorded: {args['key']} = {args['value']}"}]}

@tool(
    "recall_context",
    "Retrieve everything remembered about this customer — recent episodes and standing facts. "
    "Call this FIRST, before responding to the customer's first message in any session.",
    {"user_id": str},
)
async def recall_context(args):
    bucket = _bucket(args["user_id"])
    if not bucket["episodic"] and not bucket["semantic"]:
        return {"content": [{"type": "text", "text": "No prior history for this customer."}]}
    # Last 5 episodes only — a real system would page or summarize instead of returning
    # everything; this cap is a teaching simplification, not a designed retention policy.
    episodes = "\n".join(f"- [{e['channel']}] {e['note']}" for e in bucket["episodic"][-5:])
    facts = "\n".join(f"- {k}: {v}" for k, v in bucket["semantic"].items())
    return {"content": [{"type": "text",
             "text": f"Recent episodes:\n{episodes or '(none)'}\n\nStanding facts:\n{facts or '(none)'}"}]}

memory_tools_server = create_sdk_mcp_server(
    name="memory_tools", version="1.0.0",
    tools=[remember_episode, remember_fact, recall_context],
)
print("remember_episode, remember_fact, recall_context registered.")


### Step 6 — Channel adapters, and the amnesia failure

`chat_adapter` and `sms_adapter` are deliberately thin: each takes a raw customer message and a
`user_id` and returns a normalized `{"user_id", "channel", "text"}` shape. That's the entire
contract a channel adapter has to satisfy for everything downstream — the agent, the memory tools —
to not care which channel it came from. **A production adapter does far more** — auth, rate limits,
formatting a reply to fit an SMS's length limit, mapping a platform-specific user id to your
internal `user_id` — today's version is deliberately just the normalization contract, the same
simplification Day 1 used for `score()` standing in for real retrieval.

**Why `user_id` has to be threaded into the prompt text itself, explicitly, every time:** in a real
deployment the customer is already authenticated when their message arrives — the channel already
knows who they are — and that identity gets attached to the request as metadata, not typed by the
customer. This lab has no separate metadata channel into the agent, so the adapter's `user_id` gets
folded directly into the message text below (`[customer_id: ...]`) as a stand-in for that
attachment. Leave it out and the agent has no stable identity to key `remember_episode` or
`recall_context` on — it will invent a different placeholder id each session, and cross-session
memory silently breaks even with a perfectly-worded system prompt, because the *key* was never the
same key twice. This is worth treating as a real failure mode, not a notebook footnote: it's an easy
bug to ship in production too, if a channel integration authenticates the customer but forgets to
actually pass that identity into the agent's input.

`weak_memory_options` wires all three memory tools in, but — same shape of gap as every other
"weak" agent this week — the system prompt only says *"You can remember and recall customer
context if useful."* Nothing requires calling `recall_context` first.

**What to expect, and why:** session 1 (chat) has the customer dispute a $45 charge; the agent may
or may not call `remember_episode` — it's optional, so it's inconsistent. Session 2 is a **brand
new `ClaudeSDKClient`**, on the `sms` channel, same `user_id`, asking *"Any update on my dispute?"*
Because it's a new session, there's zero shared context unless the agent explicitly calls
`recall_context` — and the weak prompt doesn't require it. Expect the agent to ask the customer to
re-explain what dispute they mean, as if session 1 never happened.

🔍 **Try it yourself** with the two-session flow below before changing anything — the amnesia needs
to reproduce for the fix to land.

In [ ]:
def chat_adapter(user_id: str, text: str) -> dict:
    """Stand-in for a real chat-widget SDK integration. Its only job: normalize
    into the shape the agent and memory tools share, regardless of source channel."""
    return {"user_id": user_id, "channel": "chat", "text": text}

def sms_adapter(user_id: str, text: str) -> dict:
    """Stand-in for a real SMS gateway webhook (e.g. Twilio). Same normalized shape as chat —
    that's the whole point: the agent never has to know which adapter a message came through."""
    return {"user_id": user_id, "channel": "sms", "text": text}

# "if useful" is the bug under test — memory tools are wired in, but nothing REQUIRES
# recall_context to be called. Same shape of gap as every "weak" agent this week.
weak_memory_options = ClaudeAgentOptions(
    system_prompt=(
        "You are a banking customer service assistant. You can remember and recall customer "
        "context if useful."
    ),
    mcp_servers={"memory_tools": memory_tools_server},
    allowed_tools=["mcp__memory_tools__remember_episode",
                   "mcp__memory_tools__remember_fact",
                   "mcp__memory_tools__recall_context"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

user_id = "cust-771"

# customer_id is threaded into the prompt text explicitly (see the adapter markdown above)
# — the agent has no other way to know which memory_store bucket belongs to this customer.
print("--- session 1 (chat) — weak_memory_options ---")
msg1 = chat_adapter(user_id, "I want to dispute a $45 charge on my card ending 9931, it's not mine.")
await ask(weak_memory_options,
          f"[customer_id: {msg1['user_id']}] [channel: {msg1['channel']}] {msg1['text']}")

# ask() opens a BRAND NEW ClaudeSDKClient here — this is a genuinely separate session,
# not a continued conversation, which is what makes this a real test of cross-session
# memory rather than the model just remembering turn 1 from its own context window.
print("\n--- session 2 (sms), NEW ClaudeSDKClient, same customer — weak_memory_options ---")
msg2 = sms_adapter(user_id, "Any update on my dispute?")
await ask(weak_memory_options,
          f"[customer_id: {msg2['user_id']}] [channel: {msg2['channel']}] {msg2['text']}")


### Fixing it — recall first, remember before ending

`memory_options` adds two hard requirements: **call `recall_context` before responding to the
customer's first message in a session**, and **call `remember_episode` before ending any turn where
the customer revealed something worth recalling.** Neither is optional this time.

Re-run the identical two-session flow. Expect session 1 to log the dispute as an episode; session 2
— still a brand new `ClaudeSDKClient`, still a different channel — to call `recall_context` first,
retrieve the chat-channel note about the $45 dispute, and respond without asking the customer to
re-explain anything.

**What this proves, and what it doesn't:** this proves memory persistence lives in `memory_store`,
external to any one session — not that the SDK has some built-in cross-session memory feature. If
`memory_store` were an in-memory Python dict in a real deployment, it would reset on every process
restart; production versions of this pattern back it with a real database. The *mechanism* — recall
before responding, remember before ending, keyed by a stable `user_id` — is what's real; today's
storage is a teaching simplification, same as `policy_chunks` being a list instead of a vector
store.

✅ **Checkpoint check** — before moving on, confirm session 2's response references the actual
dispute detail from session 1 (the $45 amount or the card ending 9931), not a generic "can you tell
me more."


In [ ]:
memory_options = ClaudeAgentOptions(
    system_prompt=(
        "You are a banking customer service assistant. Every message arrives tagged with "
        # Binds the agent to the customer_id given in the prompt tag, rather than letting
        # it invent one — this is the fix for the "different key each session" failure mode.
        "'[customer_id: ...]' — that is the customer's stable identity; always pass it as the "
        "user_id argument to every memory tool call, exactly as given, never invent or alter it. "
        # Mandatory recall-first, same move as Day 1's "you MUST call kb_search first."
        "Before responding to the customer's FIRST message in this session, you MUST call "
        "recall_context with that user_id to check for prior history — never skip this, even if "
        "the message seems self-contained. If the customer tells you something worth remembering "
        "(a dispute, a preference, a standing issue), call remember_episode with that same user_id "
        "before ending your turn. Use recalled context naturally — never make the customer repeat "
        "something they already told you on another channel."
    ),
    mcp_servers={"memory_tools": memory_tools_server},
    allowed_tools=["mcp__memory_tools__remember_episode",
                   "mcp__memory_tools__remember_fact",
                   "mcp__memory_tools__recall_context"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# A different user_id than the weak demo — keeps the two runs' memory_store entries
# from colliding, so both demos stay independently inspectable.
user_id_2 = "cust-882"

print("--- session 1 (chat) — memory_options ---")
msg1 = chat_adapter(user_id_2, "I want to dispute a $45 charge on my card ending 9931, it's not mine.")
await ask(memory_options,
          f"[customer_id: {msg1['user_id']}] [channel: {msg1['channel']}] {msg1['text']}")

# Same "brand new session" shape as the weak demo — the only variable that changed
# between this cell and the last one is the SYSTEM PROMPT, not the session mechanics.
print("\n--- session 2 (sms), NEW ClaudeSDKClient, same customer — memory_options ---")
msg2 = sms_adapter(user_id_2, "Any update on my dispute?")
await ask(memory_options,
          f"[customer_id: {msg2['user_id']}] [channel: {msg2['channel']}] {msg2['text']}")

# The real pass/fail check: does memory_store actually hold this customer's episode now?
# (Checking the printed conversation text alone isn't enough — see the markdown above.)
print("\nmemory_store for", user_id_2, ":", json.dumps(memory_store[user_id_2], indent=2))


### Semantic memory — distilling a pattern, not just logging one

The cell below pre-seeds two more dispute episodes for `cust-882`, simulating a customer who has now
disputed three separate charges in 90 days, then asks the agent to write a **standing fact**, not
another episode.

**Why this is a `remember_fact` call and not a fourth `remember_episode`:** "customer disputed a
charge on 2026-07-20" is a fact about *what happened*. "Repeat disputer — 3 disputes in 90 days" is
a fact *about the customer*, derived by looking across several episodes — useful to a risk model or
a routing decision in a way that scanning the raw episodic log every time wouldn't be. This is the
distinction Step 5 introduced in the abstract; this is what it looks like as an actual tool call.

🔍 **Try it yourself:** ask the agent directly — *"Based on this customer's history, should we flag
their account for review?"* — and see whether it reasons from the semantic fact (fast, direct) or
re-reads the raw episodic log to re-derive the same conclusion (slower, and a sign the semantic
layer isn't being trusted once it exists).

✅ **Checkpoint — Lab 2 complete.** Memory persists across a channel switch, backed by a real
two-session test, with episodic and semantic memory doing visibly different jobs. Save the notebook
here — **this is today's ship line.**


In [ ]:
# Pre-seed two more episodes to simulate a repeat-dispute pattern, then ask the agent
# to distill a semantic fact from the pattern — no API call needed for the seeding itself,
# this is plain Python mutating the same memory_store dict the tools use.
bucket = _bucket(user_id_2)
bucket["episodic"].append({"channel": "chat", "note": "Disputed a $12 charge, card ending 9931.",
                            "ts": time.time()})
bucket["episodic"].append({"channel": "sms", "note": "Disputed a $60 charge, card ending 9931.",
                            "ts": time.time()})
# cust-882 now has 3 episodic entries total (1 from the cell above + these 2) — enough
# to make "repeat disputer" a pattern worth distilling, not a one-off.

# customer_id still has to be threaded in explicitly — ask() opens a fresh session with
# no prior turns, so there is nothing else in this call that identifies WHICH customer.
await ask(memory_options,
          f"[customer_id: {user_id_2}] Based on this customer's dispute history, should we record "
          "a standing risk flag? If so, record it as a fact.")

# Expect ONE new key under bucket["semantic"] — a distillation, not a 4th episodic entry.
print("\nSemantic facts for", user_id_2, ":", memory_store[user_id_2]["semantic"])


---

## Lab 3 — Retail: agent-assist mode, watched by a QA hook

**Ship criterion:** the agent drafts a suggested reply for a human agent to review — never sends it
to the customer directly — and every human decision (accept, edit, reject) gets logged by a hook
that fires on the *human's* action, not on the agent's say-so.

### Step 7 — Agent-assist is a prompt contract, not a new mechanism

The retail knowledge base below is the same shape as Day 1's Insurance KB and Lab 1's policy
specialist — clauses, `kb_search`, grounding rules. **Nothing new is being built here.** What makes
this "agent-assist mode" instead of a resolution agent is entirely in the system prompt: the agent
is told it is drafting for a human, not answering a customer, and to self-report a confidence level.

**Why this is worth calling out as *not* a new mechanism:** it would be easy to assume agent-assist
needs a fundamentally different architecture — a different tool-calling loop, a human-approval gate
baked into the SDK, something like that. It doesn't. The grounding pattern from Day 1 and Lab 1 is
identical; only the *audience and output contract* changed. That's a genuinely reusable insight: the
same grounded-agent mechanism can sit behind a fully autonomous resolution agent (Day 1), a routed
specialist (Lab 1), or a human-reviewed draft (this lab) — the difference is what happens to the
output after the agent produces it, not how the agent produces it.

`weak_assist_options` skips the grounding mandate, on purpose, to reproduce a familiar failure in a
new setting. Ask about returning a blender bought 20 days ago — expect a confident, plausible,
**wrong** answer ("most retailers allow 30 days"), because nothing requires `kb_search` first.

🔍 **Try it yourself** with the blender question below before applying the fix.


In [ ]:
return_chunks = [
    {"id": "RET-1.1", "text": "Standard merchandise may be returned within 30 days of purchase "
     "with a valid receipt, unmarked and in original packaging."},
    # RET-1.2 is the scripted-failure clause: a 14-day exception that overrides the 30-day
    # standard above for electronics specifically — same "base rule + overriding exception"
    # shape as Day 1's POL-4.2/POL-9.1 pair.
    {"id": "RET-1.2", "text": "Electronics (including headphones, speakers, and small appliances) "
     "must be returned within 14 days of purchase; the 30-day standard window does not apply."},
    {"id": "RET-1.3", "text": "A 15% restocking fee applies to any opened electronics return."},
    {"id": "RET-1.4", "text": "Clearance and final-sale items, marked as such at purchase, are "
     "not eligible for return under any circumstances."},
]

def retail_search(query: str, top_k: int = 3):
    # Reuses score() from Lab 1's cell — same scoring function, different document set.
    ranked = sorted(return_chunks, key=lambda c: score(query, c["text"]), reverse=True)
    return ranked[:top_k]

@tool(
    "kb_search",   # same tool NAME as Lab 1's policy kb_search — fine, they live on
                    # different MCP servers (policy_tools vs retail_tools below), so the
                    # fully-qualified mcp__<server>__kb_search names never collide.
    "Search the retail return-policy knowledge base for clauses relevant to a customer's "
    "question. Always call this before drafting any reply about returns or refunds.",
    {"query": str},
)
async def retail_kb_search(args):
    results = retail_search(args["query"])
    formatted = "\n".join(f"[{r['id']}] {r['text']}" for r in results)
    return {"content": [{"type": "text", "text": formatted}]}

retail_tools_server = create_sdk_mcp_server(
    name="retail_tools", version="1.0.0", tools=[retail_kb_search],
)

# No "MUST call kb_search" mandate — the bug under test, same shape as every other
# "weak" agent this week, just relocated to the assist-mode framing.
weak_assist_options = ClaudeAgentOptions(
    system_prompt=(
        "You are drafting a suggested reply for a HUMAN support agent to review before it is "
        "sent — you are not talking to the customer directly. Write the suggested customer-"
        "facing reply as your output."
    ),
    mcp_servers={"retail_tools": retail_tools_server},
    allowed_tools=["mcp__retail_tools__kb_search"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Expect a confident, plausible, WRONG answer — "most retailers allow 30 days" — because
# nothing forces retrieval of RET-1.2's actual 14-day electronics exception.
await ask(weak_assist_options,
          "Customer message: \"Can I return a blender I bought 20 days ago?\"")


### The fix — same grounding discipline as everywhere else this week

`grounded_assist_options` adds exactly the grounding mandate Day 1 and Lab 1 both used, plus one
addition specific to this mode: **end every draft with a self-reported confidence** (high / medium /
low), so the human reviewing it has a signal for how carefully to check it, not just the draft
itself.

Run the identical blender question. Expect `kb_search` to retrieve RET-1.2 (14-day electronics
window, not the 30-day standard), a correct draft stating the return is likely too late, and a
confidence level at the end.

**Why confidence is self-reported rather than computed:** a real production system might derive
confidence from retrieval score, clause specificity, or historical accuracy — today's version asks
the model to self-report it, which is a real limitation worth naming, not hiding: a model's
self-reported confidence is a plausible-sounding number, not a calibrated probability. It's useful
as a triage signal for a human reviewer ("check this one more carefully"), not as a metric to
trust on its own — which is exactly why Step 8's QA hook exists: to check the suggestion against
what actually happened, the same discipline as Day 1 Lab 3's "don't trust the agent's self-report."


In [ ]:
grounded_assist_options = ClaudeAgentOptions(
    system_prompt=(
        "You are drafting a suggested reply for a HUMAN support agent to review before it is "
        "sent — you are not talking to the customer directly. For ANY question about returns or "
        # Same grounding mandate as every other fixed agent this week — no new mechanism.
        "refunds, you MUST call kb_search first and draft your reply only from the returned "
        "clauses, citing the clause id(s). If the clauses don't clearly answer the question, say "
        "so in the draft rather than guessing. End every draft with a line: "
        # New relative to a plain resolution agent: a self-reported confidence line, since
        # the audience is a human reviewer deciding how carefully to check the draft.
        "'Confidence: high / medium / low' based on how directly the retrieved clauses answer "
        "the question."
    ),
    mcp_servers={"retail_tools": retail_tools_server},
    allowed_tools=["mcp__retail_tools__kb_search"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Same question as the weak version — expect kb_search called, RET-1.2 retrieved, and a
# correct draft ("too late, 14-day window") ending in a Confidence line.
await ask(grounded_assist_options,
          "Customer message: \"Can I return a blender I bought 20 days ago?\"")


### Step 8 — QA hooks: logged by the human's action, not the agent's

This is a deliberate contrast with Day 1 Lab 3. There, `log_outcome` was a tool the *agent* called
on itself, at the end of its own turn — a self-report. Here, `log_assist_review` is a **plain Python
function, not an `@tool`** — it's never wired into any agent's `allowed_tools`, and the agent has no
way to call it and no visibility into it. It fires when the surrounding application records what the
*human* actually did with a draft: accepted as-is, edited before sending, or rejected outright.

**Why this distinction is the actual teaching point of "QA hooks," not a stylistic choice:** a
self-reported log can only ever be as honest as the agent producing it — if the agent is confused
about whether it resolved something, its own log entry inherits that confusion (Day 1's whole Lab 3
point). A hook wired to the human's action can't be spoofed by the agent, confused or otherwise,
because the agent was never in that part of the loop at all. That's a structurally stronger
guarantee than asking an agent to grade its own homework, and it's the reason production QA/eval
pipelines for assist surfaces are usually built this way — instrumented at the UI/application layer,
not inside the agent's own tool-calling loop.

Run the three simulated reviews below: a good grounded draft the human accepts as-is; the earlier
ungrounded blender draft the human catches and rejects; the same question re-asked against
`grounded_assist_options`, now accepted.

✅ **Checkpoint — Lab 3 complete.** Assist drafts are grounded, confidence-scored, and every human
decision is logged by a hook the agent can't see or influence. Save the notebook here.


In [ ]:
qa_log = []

def log_assist_review(suggested_text: str, human_action: str, final_text: str, note: str = ""):
    """Called by the surrounding application/UI when a human resolves a draft — NOT by the
    agent. human_action is one of: accepted | edited | rejected."""
    # This is a PLAIN Python function — deliberately never wrapped in @tool, never put in
    # any allowed_tools list. The agent has no way to call it, see it, or influence its
    # contents. That's the entire mechanism that makes this hook trustworthy: it fires on
    # what a human actually did, not on what the agent claims about its own draft.
    assert human_action in ("accepted", "edited", "rejected")
    entry = {"human_action": human_action, "suggested_text": suggested_text,
             "final_text": final_text, "note": note, "ts": time.time()}
    qa_log.append(entry)
    return entry

# Review 1 — a good grounded draft, accepted as-is. In a real app this call would be
# triggered by a human clicking "send" in a review UI, not by notebook code — here we're
# simulating that click directly to keep the lab self-contained.
good_draft = ("Unfortunately headphones fall under our 14-day electronics return window "
              "[RET-1.2], and it's been 20 days, so this one's past the window. "
              "Confidence: high")
log_assist_review(good_draft, "accepted", good_draft, "Correct citation, sent as-is.")

# Review 2 — the earlier ungrounded blender draft (from the weak_assist_options cell),
# caught and rejected by a human before it ever reached the customer.
bad_draft = "Sure — most retailers allow returns within 30 days, so you're still within the window."
corrected = ("Actually, small appliances fall under our 14-day electronics window [RET-1.2], "
             "and 20 days has passed, so this one can't be returned.")
log_assist_review(bad_draft, "rejected", corrected,
                   "No citation, wrong window assumed — human caught it before it went out.")

# Review 3 — re-ask the same question against the grounded version, then log the
# now-correct result. Reusing _run_specialist (Lab 1) instead of ask() here is the fix,
# not a stylistic choice: ask() only prints, it never returns a value, so an earlier
# draft of this cell hand-authored fixed_draft below the print — meaning the qa_log
# entry always said "accepted" regardless of what the live call actually produced.
# _run_specialist captures the model's real final text, so the log now reflects what
# was actually said, not what the notebook author expected it to say.
print("--- re-run against grounded_assist_options ---")
fixed_draft = await _run_specialist(
    grounded_assist_options,
    "Customer message: \"Can I return a blender I bought 20 days ago?\""
)
print(fixed_draft)

log_assist_review(fixed_draft, "accepted", fixed_draft, "Grounded version, correct on first try.")

# Expect exactly 3 entries, in order: accepted, rejected, accepted.
print("\nqa_log:")
for entry in qa_log:
    print(f"[{entry['human_action']}] {entry['note']}")


## Closing out

Three labs, one shared discipline underneath the surface topics (routing, memory, assist): **state
has to survive a handoff.** Lab 1's handoff is supervisor → specialist — the customer's question has
to arrive at the right specialist intact, and a multi-part question has to come back synthesized,
not half-answered. Lab 2's handoff is session → session across a channel switch — the customer's
prior context has to survive a boundary the SDK does not bridge automatically. Lab 3's handoff is
agent → human — a draft has to survive being reviewed, and what happens to it in review has to be
captured by something the agent itself can't touch.

Before wrapping up, look back at your own test conversations from today: for Lab 1, could you tell
whether the supervisor routed correctly just by reading the final answer, or did you have to check
which tools actually got called? For Lab 2, did session 2 actually use the recalled context, or did
it just not-ask a redundant question (a weaker bar)? For Lab 3, does the QA log tell you something
the agent's own drafts couldn't have told you on their own? If the answer to any of these is "I'm
not sure," that's worth resolving now, out loud, rather than assuming a working demo means the
underlying guarantee actually held.

---

## Ship rubric

| Checkpoint | Pass criteria |
|---|---|
| Lab 1 | A single-domain question routes to exactly one specialist — not both, not neither |
| Lab 1 | A question spanning both domains triggers both specialists, synthesized into one answer |
| Lab 2 | Session 2 (different channel, new `ClaudeSDKClient`) recalls session 1's specific detail without the customer repeating it |
| Lab 2 | A semantic fact is written as a distillation across episodes — not a fourth raw episode |
| Lab 3 | A draft is never sent to the customer directly — it's grounded, cited, and confidence-scored for a human reviewer |
| Lab 3 | `log_assist_review` is called by the harness/UI, never by the agent — check `allowed_tools` doesn't include it |

## Known-breakage cheat sheet

| Symptom | Likely cause | Fix |
|---|---|---|
| Supervisor answers directly, never calls a specialist tool | System prompt allows "if needed" instead of mandating routing | Tighten to "you do not answer these questions yourself" |
| Supervisor calls only one specialist on a two-part question | Prompt doesn't explicitly name the multi-intent case | Add "if a question needs both, call both and synthesize" |
| `_run_specialist` returns an empty string | Specialist's `AssistantMessage` had no `TextBlock` content, or the wrong message type was checked | Print the raw `message` objects once to confirm the shape before filtering |
| Session 2 doesn't recall session 1's context | `recall_context` isn't mandated as the first call of a session, or `user_id` differs between sessions | Check the system prompt's "MUST call recall_context first" clause and that both adapters were given the same `user_id` |
| `remember_fact` never fires for the repeat-disputer case | Prompt doesn't distinguish "record a fact" from "record an episode" | Ask explicitly for a standing fact, or tighten the prompt's fact-vs-episode language |
| Assist draft gets sent straight to the customer in your own extension | Nothing in this lab enforces a human step — that enforcement lives in the surrounding application, not the agent | The agent's contract ("drafting for a human") is a prompt convention; the actual gate has to be built into whatever UI/queue displays the draft |
| `log_assist_review` raises an `AssertionError` | `human_action` wasn't one of `accepted \| edited \| rejected` | Check the literal string passed in |

---

## Appendix — the same three patterns in other lanes

**Supervisor + specialists (Lab 1's pattern):** any domain with two genuinely different question
types splits the same way. Banking: a disputes specialist vs. a card-services specialist. Retail: an
order-status specialist vs. a returns specialist. The supervisor's routing rule and the
`_run_specialist` helper don't change — only which specialist options get built and what each one's
tool does.

**Memory across channels (Lab 2's pattern):** the episodic/semantic split and the recall-first /
remember-before-ending discipline are domain-independent. Insurance: episodic = "asked about rental
coverage on 2026-07-18," semantic = "prefers email over phone." Retail: episodic = "asked about a
return on chat," semantic = "frequent returner — flag for a lighter-touch policy." This is also
directly the pattern from this morning's Telecom hands-on (cross-session memory) — same mechanism,
this lab just gives it a full two-channel, two-session proof instead of a single-session preview.

**Agent-assist + QA hook (Lab 3's pattern):** the draft-for-a-human contract and the
human-action-triggered hook generalize to any domain where full autonomy isn't the goal — a human
agent overseeing many conversations at once, a compliance-sensitive vertical where every reply needs
sign-off, or a new agent still being trusted incrementally before it's given direct-to-customer
access.


---

## Appendix — the same supervisor pattern in LangGraph and CrewAI

Lab 1 was built on the Claude Agent SDK, on purpose — one continuous build thread for the week.
This section exists for **framework literacy**: the same supervisor-routes-to-specialists problem
(CLM-1077's denial, the same policy KB and claims database from Lab 1), rebuilt twice more —
once in [LangGraph](https://langchain-ai.github.io/langgraph/), once in
[CrewAI](https://docs.crewai.com/) — because "multi-agent orchestration" isn't one pattern, it's a
family, and these two frameworks make different parts of it explicit by design. It's optional —
skip it if you're short on time, nothing later in the week depends on it.

**Setup:** `pip install langgraph langchain langchain-anthropic crewai`. Same authentication note
as anywhere else a raw `ChatAnthropic`/`LLM` object is used: both frameworks talk to the Anthropic
API directly and need a real `ANTHROPIC_API_KEY` environment variable — the Claude Code CLI login
that authenticates the rest of this notebook doesn't cover them.

### LangGraph — explicit graph routing

The Claude Agent SDK's supervisor pattern (`_run_specialist`, Lab 1 Steps 3–4) makes routing a
**tool call**: the supervisor's LLM decides to call `ask_policy_specialist` or
`ask_claims_specialist`, the same way it would call any other tool. LangGraph makes routing a
**graph edge** instead: a dedicated node decides which node(s) run next, and that decision is a
first-class part of the graph's structure, not a tool the model happens to pick. Neither is more
"correct" — they're different places to put the same decision. LangGraph's version is what you
reach for when the routing logic itself needs to be inspectable, testable, or visualized
independently of any one LLM call — which is exactly what the offline check further down proves.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool as lc_tool
from langchain_anthropic import ChatAnthropic

lg_model = ChatAnthropic(model="claude-sonnet-4-6")

# Reuses Lab 1's search()/policy_chunks and claims_db verbatim — same KB, same claims data,
# only the tool-definition syntax changes (docstring + type hints, LangChain's convention).
@lc_tool
def lg_kb_search(query: str) -> str:
    """Search the insurance policy knowledge base for clauses relevant to a customer's question."""
    results = search(query)
    return "\n".join(f"[{r['id']}] {r['text']}" for r in results)

@lc_tool
def lg_claim_lookup(claim_id: str) -> str:
    """Look up the status of an existing insurance claim by its claim id."""
    record = claims_db.get(claim_id)
    return json.dumps(record) if record else f"No claim found with id {claim_id}."

# Each specialist is a create_agent instance, same as Day 1's LangChain appendix — a
# complete, independent agent, unaware the other one (or a supervisor) exists.
lg_policy_specialist = create_agent(
    model=lg_model, tools=[lg_kb_search],
    system_prompt=(
        "You are the POLICY specialist. For ANY question about coverage, exclusions, or policy "
        "terms, you MUST call lg_kb_search first and answer only from the returned clauses, "
        "citing clause id(s) in the form [POL-x.x]."
    ),
)
lg_claims_specialist = create_agent(
    model=lg_model, tools=[lg_claim_lookup],
    system_prompt=(
        "You are the CLAIMS specialist. For ANY question about an existing claim's status, "
        "amount, or outcome, you MUST call lg_claim_lookup first and answer only from the "
        "returned record."
    ),
)

print("lg_policy_specialist and lg_claims_specialist ready.")


In [ ]:
from typing import Annotated, Literal, TypedDict

from pydantic import BaseModel
from langgraph.graph import END, START, StateGraph


# route_to is a Pydantic-typed structured output, not a tool call — a different LLM
# interaction pattern worth knowing alongside tool-calling: the model is asked to return
# data matching this schema directly, rather than deciding to invoke a named function.
class RoutingDecision(BaseModel):
    route_to: Literal["policy", "claims", "both"]
    reasoning: str


def merge_answers(a: dict, b: dict) -> dict:
    # Reducer: when policy_node and claims_node BOTH write to state["answers"] in the same
    # superstep (the "both" fan-out case), LangGraph needs to know how to combine two
    # concurrent writes to the same key instead of raising a conflict — this merges them.
    return {**a, **b}


class SupervisorState(TypedDict):
    question: str
    route: Literal["policy", "claims", "both"]
    answers: Annotated[dict[str, str], merge_answers]
    final: str


router = lg_model.with_structured_output(RoutingDecision)

def supervisor_node(state: SupervisorState) -> dict:
    decision = router.invoke(
        f"Question: {state['question']}\n\n"
        "route_to 'policy' for coverage/exclusion/terms questions, 'claims' for existing "
        "claim status/amount/outcome questions, 'both' if the question genuinely needs "
        "information from both (e.g. why a claim was denied AND what the underlying rule is)."
    )
    return {"route": decision.route_to}

def policy_node(state: SupervisorState) -> dict:
    result = lg_policy_specialist.invoke({"messages": [{"role": "user", "content": state["question"]}]})
    return {"answers": {"policy": result["messages"][-1].content}}

def claims_node(state: SupervisorState) -> dict:
    result = lg_claims_specialist.invoke({"messages": [{"role": "user", "content": state["question"]}]})
    return {"answers": {"claims": result["messages"][-1].content}}

def synthesize_node(state: SupervisorState) -> dict:
    parts = [f"{k.title()} specialist: {v}" for k, v in state["answers"].items()]
    return {"final": "\n\n".join(parts)}

def route_after_supervisor(state: SupervisorState) -> list[str]:
    # Returning a LIST of node names is what triggers a fan-out: for "both", policy_node
    # and claims_node run in the SAME superstep, in parallel, not one after the other.
    if state["route"] == "both":
        return ["policy_node", "claims_node"]
    return ["policy_node"] if state["route"] == "policy" else ["claims_node"]


graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("policy_node", policy_node)
graph.add_node("claims_node", claims_node)
graph.add_node("synthesize", synthesize_node)
graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route_after_supervisor, ["policy_node", "claims_node"])
# Both branches point at the SAME node. LangGraph runs synthesize once, after whichever
# subset of {policy_node, claims_node} actually executed that step — single-route
# questions reach it with one answer key, "both" reaches it with two, already merged.
graph.add_edge("policy_node", "synthesize")
graph.add_edge("claims_node", "synthesize")
graph.add_edge("synthesize", END)
supervisor_graph = graph.compile()

# The CLM-1077 multi-intent question from Lab 1 — same test, same expected fan-out.
result = supervisor_graph.invoke({
    "question": "My claim CLM-1077 got denied — why, and what's the actual filing deadline?",
    "answers": {},
})
print("route:", result["route"])
print("\nfinal:\n", result["final"])


### Offline wiring check — no API key needed

This doesn't test whether `router` makes the right routing decision — that needs a real key, and
is exactly the same kind of judgment call Lab 1's supervisor makes. What it proves is narrower and
purely mechanical: that the graph's **fan-out and fan-in actually work** — that a `"both"` route
really does run `policy_node` and `claims_node` in parallel, that a single route runs only one of
them, and that `synthesize` correctly receives whichever answers actually landed either way. This
is the part of a graph-based supervisor that's worth testing independently of any LLM call, and
LangGraph's structure makes that possible in a way a pure tool-calling loop doesn't — you can swap
in deterministic stand-ins for every node and exercise the routing logic on its own.


In [ ]:
# Same graph SHAPE as above, with deterministic stand-in nodes instead of real LLM calls —
# proves the routing/fan-out/fan-in mechanics, not the model's judgment.
def _stub_supervisor(state: SupervisorState) -> dict:
    q = state["question"]
    if "CLM-1077" in q and "deadline" in q:
        return {"route": "both"}
    return {"route": "claims"} if "CLM-1077" in q else {"route": "policy"}

def _stub_policy(state: SupervisorState) -> dict:
    return {"answers": {"policy": "POL-2.5: 7-day filing window."}}

def _stub_claims(state: SupervisorState) -> dict:
    return {"answers": {"claims": "CLM-1077 denied: late incident report."}}

_test_graph = StateGraph(SupervisorState)
_test_graph.add_node("supervisor", _stub_supervisor)
_test_graph.add_node("policy_node", _stub_policy)
_test_graph.add_node("claims_node", _stub_claims)
_test_graph.add_node("synthesize", synthesize_node)   # the real synthesize_node — not under test, just reused
_test_graph.add_edge(START, "supervisor")
_test_graph.add_conditional_edges("supervisor", route_after_supervisor, ["policy_node", "claims_node"])
_test_graph.add_edge("policy_node", "synthesize")
_test_graph.add_edge("claims_node", "synthesize")
_test_graph.add_edge("synthesize", END)
_test_compiled = _test_graph.compile()

r1 = _test_compiled.invoke({"question": "does my policy cover a rental car?", "answers": {}})
assert set(r1["answers"].keys()) == {"policy"}, r1

r2 = _test_compiled.invoke({"question": "what's the status of CLM-1077?", "answers": {}})
assert set(r2["answers"].keys()) == {"claims"}, r2

r3 = _test_compiled.invoke({
    "question": "My claim CLM-1077 got denied — why, and what's the actual filing deadline?",
    "answers": {},
})
assert set(r3["answers"].keys()) == {"policy", "claims"}, r3

print("single-route (policy only):", r1["answers"])
print("single-route (claims only):", r2["answers"])
print("fan-out (both, merged):", r3["answers"])
print("\nOK: single-route and fan-out/fan-in with reducer-merged state all wired correctly.")


### CrewAI — role-based delegation

A third way to place the same routing decision. The Claude Agent SDK makes it a tool call;
LangGraph makes it a graph edge; CrewAI makes it a **manager agent's delegation decision** —
you don't write the routing logic at all. You describe each specialist's `role`/`goal`/`backstory`,
put them in a `Crew` with `process=Process.hierarchical`, and CrewAI's manager (driven by
`manager_llm`) decides which agent(s) to delegate the task to, based on how well each agent's
role matches what the task needs. This is the most implicit of the three — less code, less control,
more trust placed in the framework's own delegation judgment.

**Same domain, same CLM-1077 test case, same underlying tools** — only the orchestration
philosophy changes.


In [ ]:
from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool as crew_tool

# CrewAI's LLM uses a LiteLLM-style "provider/model" string, not a ChatAnthropic object —
# a third way of pointing at the same underlying Claude model.
crew_llm = LLM(model="anthropic/claude-sonnet-4-6")

@crew_tool("kb_search")
def crew_kb_search(query: str) -> str:
    """Search the insurance policy knowledge base for clauses relevant to a customer's question."""
    results = search(query)   # Lab 1's search()/policy_chunks, reused again
    return "\n".join(f"[{r['id']}] {r['text']}" for r in results)

@crew_tool("claim_lookup")
def crew_claim_lookup(claim_id: str) -> str:
    """Look up the status of an existing insurance claim by its claim id."""
    record = claims_db.get(claim_id)
    return json.dumps(record) if record else f"No claim found with id {claim_id}."

# allow_delegation=False on the SPECIALISTS: they answer, they don't delegate further —
# only the (implicit) manager agent, created by process=Process.hierarchical below, delegates.
policy_agent = Agent(
    role="Policy Specialist",
    goal="Answer questions about policy coverage, exclusions, and terms using kb_search, "
         "citing clause id(s), never guessing.",
    backstory="You handle coverage and policy-term questions for an insurance CX team.",
    tools=[crew_kb_search],
    llm=crew_llm,
    allow_delegation=False,
    verbose=False,
)
claims_agent = Agent(
    role="Claims Specialist",
    goal="Answer questions about existing claims' status, amount, or outcome using "
         "claim_lookup, never guessing.",
    backstory="You handle claim status and outcome questions for an insurance CX team.",
    tools=[crew_claim_lookup],
    llm=crew_llm,
    allow_delegation=False,
    verbose=False,
)

# ONE task, NO agent assigned directly — in a hierarchical crew, the manager decides who
# (one agent, or several) actually works the task, based on the agents' role/goal text above.
routing_task = Task(
    description=(
        "Answer this customer question completely, delegating to the Policy Specialist "
        "and/or Claims Specialist as needed — use both if the question requires it: "
        "{question}"
    ),
    expected_output=(
        "A complete, accurate answer citing what each specialist found, addressing every "
        "part of the question."
    ),
)

crew = Crew(
    agents=[policy_agent, claims_agent],
    tasks=[routing_task],
    process=Process.hierarchical,
    manager_llm=crew_llm,
    verbose=True,
)

# kickoff_async, not kickoff: a Jupyter kernel already runs its own asyncio event loop
# (the same reason top-level `await` works in this notebook at all) — CrewAI's synchronous
# kickoff() detects a running loop and refuses to execute inside it, raising a RuntimeError.
# Every other multi-turn call in this notebook is already async for the same underlying
# reason; this is not a special case, just easy to miss coming from CrewAI's own docs,
# which mostly show kickoff() in plain-script examples with no event loop already running.
# The same CLM-1077 multi-intent question as every other version of this test.
crew_result = await crew.kickoff_async(inputs={
    "question": "My claim CLM-1077 got denied — why, and what's the actual filing deadline?"
})
print(crew_result)


**A gap worth naming rather than papering over:** unlike the LangChain and LangGraph cells above,
there's no offline wiring check for the CrewAI cell. LangChain/LangGraph's `BaseChatModel`
abstraction has a documented fake-model test double (`FakeMessagesListChatModel`) that plugs
straight into the same graph a real model would run in; CrewAI's `LLM` wraps LiteLLM directly and
doesn't expose an equivalent swap-in double. What *is* verified: `Agent`, `Task`, and `Crew`
construct correctly with these exact field names and types — real object-graph validation, not
just "the import doesn't crash." What's *not* verified without a real key: that
`crew.kickoff_async()` actually runs and the manager delegates sensibly. That's a genuine
capability gap in this particular framework's testability, not a corner cut for this notebook.

**One more real gap this caught:** the cell uses `crew.kickoff_async()`, not the `kickoff()` shown
in most CrewAI examples — plain-script examples don't have an event loop already running, but a
Jupyter kernel does (the same reason top-level `await` works anywhere else in this notebook), and
CrewAI's synchronous `kickoff()` refuses to run inside one. Worth remembering for your own capstone
if you reach for CrewAI inside a notebook rather than a plain script.

### Three placements for one decision

| | Where the routing decision lives | What you write |
|---|---|---|
| Claude Agent SDK (Lab 1) | A tool call the supervisor's LLM chooses to make | Two `@tool` dispatch functions, a system prompt that mandates using them |
| LangGraph | A graph edge, decided by a dedicated node | A `StateGraph` with conditional edges — the routing logic is inspectable and testable on its own |
| CrewAI | A manager agent's delegation judgment | `role`/`goal`/`backstory` per specialist — no routing code at all |

All three answer the same CLM-1077 question correctly when given a real key, because the
underlying facts (the KB, the claims database) and the underlying model are identical. What
differs is how much of the routing decision you write explicitly versus hand to the framework —
worth having an opinion on for your own capstone, not a question with one right answer.
